# Workflow Demonstration - Newstensity

**DEMO / SIMULASI:** dataset berisi 500 artikel sintetis, bukan berita aktual. Notebook ini menunjukkan validasi/deduplikasi, ekstraksi entitas, clustering TF-IDF + KMeans, dan baseline sentimen leksikon. Tidak ada klaim akurasi model.

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from data_processing import process_news
from issue_detection import enrich_issues, extract_keywords
from sentiment_analysis import analyze_sentiment
raw = pd.read_csv(ROOT / "data" / "sample_news.csv")
raw.head()

,article_id,source,source_url,published_at,collected_at,title,body,language,issue_group_simulation,sentiment_simulation,is_simulation
0,SIM-0001,Media Simulasi 10,https://media-simulasi-09.example/articles/SIM...,2026-10-01T07:00:00+00:00,2026-10-01T07:05:00+00:00,[SIMULASI] Polemik dan kritik menguat dalam pe...,SIMULASI - Artikel sintetis untuk demonstrasi ...,id,Proses dan mekanisme konstitusional,Negatif,True
1,SIM-0002,Media Simulasi 09,https://media-simulasi-09.example/articles/SIM...,2026-10-01T06:19:12+00:00,2026-10-01T06:42:12+00:00,[SIMULASI] Pernyataan sejumlah pihak terkait m...,SIMULASI - Artikel sintetis untuk demonstrasi ...,id,Tanggapan aktor politik,Netral,True
2,SIM-0003,Media Simulasi 06,https://media-simulasi-02.example/articles/SIM...,2026-10-01T08:38:24+00:00,2026-10-01T09:03:24+00:00,[SIMULASI] Harapan solusi muncul dari upaya kl...,SIMULASI - Artikel sintetis untuk demonstrasi ...,id,Pandangan pakar dan hukum,Positif,True
3,SIM-0004,Media Simulasi 05,https://media-simulasi-09.example/articles/SIM...,2026-10-01T08:57:36+00:00,2026-10-01T09:03:36+00:00,[SIMULASI] Kontroversi menjadi sorotan dalam r...,SIMULASI - Artikel sintetis untuk demonstrasi ...,id,Respons dan dinamika publik,Negatif,True
4,SIM-0005,Media Simulasi 09,https://media-simulasi-01.example/articles/SIM...,2026-10-01T08:16:48+00:00,2026-10-01T08:27:48+00:00,[SIMULASI] Pendekatan konstruktif mendapat duk...,SIMULASI - Artikel sintetis untuk demonstrasi ...,id,Proses dan mekanisme konstitusional,Positif,True


## 1. Processing dan deduplikasi
Satu salinan baris ditambahkan hanya untuk memperlihatkan exact-duplicate handling.

In [2]:
demo_input = pd.concat([raw, raw.iloc[[0]]], ignore_index=True)
clean, processing_stats = process_news(demo_input)
processing_stats

{'input_rows': 501,
 'invalid_published_at': 0,
 'removed_empty_content': 0,
 'duplicate_rows_removed': 1,
 'output_rows': 500}

## 2. Issue signals dan topic clustering
Cluster ID bersifat arbitrer dan harus ditinjau; entity extraction memakai pola sederhana.

In [3]:
enriched = enrich_issues(clean)
print("Top keywords:", extract_keywords(enriched["text_normalized"].tolist(), 15))
enriched[["article_id", "entities", "cluster_id_baseline"]].head()

c:\Users\agnim\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\agnim\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\agnim\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\agnim\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\agnim\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Top keywords: [('pembahasan', 785), ('isu', 692), ('sintetis', 500), ('demonstrasi', 500), ('workflow', 500), ('newstensity', 500), ('hipotetis', 500), ('pemakzulan', 500), ('wakil', 500), ('presiden', 500), ('gibran', 500), ('bukan', 500), ('kutipan', 500), ('ringkasan', 500), ('representasi', 500)]


,article_id,entities,cluster_id_baseline
0,SIM-0002,"Gibran, Wakil Presiden",1
1,SIM-0001,"Gibran, Wakil Presiden",3
2,SIM-0008,"Gibran, Wakil Presiden",2
3,SIM-0005,"Gibran, Wakil Presiden",3
4,SIM-0006,"Gibran, Wakil Presiden",1


## 3. Sentiment baseline
Aturan leksikon hanya demonstrasi alur. Model produksi membutuhkan dataset berlabel dan evaluasi.

In [4]:
sentiment = pd.DataFrame(analyze_sentiment(enriched["text_normalized"].tolist()))
result = pd.concat([enriched.reset_index(drop=True), sentiment], axis=1)
result["sentiment_predicted"].value_counts()

sentiment_predicted
Negatif    230
Netral     150
Positif    120
Name: count, dtype: int64

## 4. Export hasil


In [5]:
output_path = ROOT / "data" / "demo_processed_output.csv"
result.to_csv(output_path, index=False)
print(output_path)
result[["article_id", "title", "cluster_id_baseline", "sentiment_predicted"]].head()

d:\Projects\MagangHub\PT Binokular\seleksi pendaftaran\newstensity-news-monitoring-workflow\data\demo_processed_output.csv


,article_id,title,cluster_id_baseline,sentiment_predicted
0,SIM-0002,[SIMULASI] Pernyataan sejumlah pihak terkait m...,1,Netral
1,SIM-0001,[SIMULASI] Polemik dan kritik menguat dalam pe...,3,Negatif
2,SIM-0008,[SIMULASI] Kontroversi menjadi sorotan dalam r...,2,Negatif
3,SIM-0005,[SIMULASI] Pendekatan konstruktif mendapat duk...,3,Positif
4,SIM-0006,[SIMULASI] Sejumlah pihak mengapresiasi komuni...,1,Positif
